In [61]:
import xarray as xr
import rioxarray as rxr
import netCDF4 as nc
import numpy as np
# Plot the data
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.basemap import Basemap

dir = '../data/2014/01/'
input_file = f'{dir}cmems_obs-wind_glo_phy_my_l4_P1M_201401.nc'
match_file = f'{dir}2014_01_jplMURSST41_.nc'

match_data = rxr.open_rasterio(match_file)
input_data = rxr.open_rasterio(input_file)

print(match_data)

CRS = 'EPSG:4326'
match_data = match_data.rio.write_crs(CRS)
input_data = input_data.rio.write_crs(CRS)

<xarray.DataArray 'sst' (time: 1, y: 5101, x: 6901)> Size: 141MB
[35202001 values with dtype=float32]
Coordinates:
  * x            (x) float64 55kB -179.0 -179.0 -179.0 ... -110.0 -110.0 -110.0
  * y            (y) float64 41kB 61.0 60.99 60.98 60.97 ... 10.02 10.01 10.0
  * time         (time) object 8B 2014-01-16 00:00:00
    spatial_ref  int64 8B 0
Attributes: (12/70)
    latitude#actual_range:          [10. 61.]
    latitude#axis:                  Y
    latitude#ioos_category:         Location
    latitude#long_name:             Latitude
    latitude#standard_name:         latitude
    latitude#units:                 degrees_north
    ...                             ...
    long_name:                      Sea Surface Temperature Monthly Mean
    standard_name:                  sea_surface_foundation_temperature
    units:                          degree_C
    _FillValue:                     -999.0
    scale_factor:                   1.0
    add_offset:                     0.0


In [62]:
print(match_data.shape)
print(input_data.northward_wind.shape)

(1, 5101, 6901)
(1, 204, 276)


In [63]:
# Downsampling the data by averaging the values
from rasterio.enums import Resampling

print('Before reprojecting', match_data.shape)
print('Before reprojecting', input_data.northward_wind.shape)
match_data = match_data.rio.reproject_match(input_data, resampling=Resampling.average)
match_data = match_data.rio.write_crs(CRS)
print('After reprojecting', match_data.shape)
print('After reprojecting', input_data.northward_wind.shape)

match_condition = (match_data.shape == input_data.northward_wind.shape)
print(match_condition)

Before reprojecting (1, 5101, 6901)
Before reprojecting (1, 204, 276)
After reprojecting (1, 204, 276)
After reprojecting (1, 204, 276)
True


In [73]:
# Write the data to a new file with the same projection as the input data   

output_file = f'{match_file[:-3]}_reprojected.tif'
print(output_file)

match_data.rio.write_crs(CRS).rio.to_raster(output_file)


../data/2014/01/2014_01_jplMURSST41__reprojected.tif


None


# Resample Chl-a data